# MongoDB CRUD Operations

Data modifications in MongoDB are called **CRUD operations** (Create, Read, Update, Delete). Write operations target a **single collection** and are **atomic at the single-document level** — a single document's update either fully happens or doesn't, but there is no automatic atomicity across multiple documents unless you use transactions.

You interact with documents using JavaScript-based methods in the MongoDB Shell (`mongosh`) or through language drivers (Node.js, Python, Java, etc.). The method names are nearly identical across drivers, so what you learn in `mongosh` transfers directly to Express/Node.

---

## 1. Insert Operations (Create)

These methods add new documents to a collection. **If the collection doesn't exist, MongoDB creates it automatically** on first insert — no schema or `CREATE TABLE` step needed.

| Method | Purpose |
|---|---|
| `insertOne()` | Adds exactly one document |
| `insertMany()` | Adds an array of multiple documents |

### The `_id` field

Every document requires a unique `_id`. If you omit it, MongoDB generates an `ObjectId` automatically — a 12-byte value that encodes a timestamp, so `_id`s are roughly sortable by creation time.

```javascript
// Insert a single user
db.users.insertOne({
  name: "Alice",
  age: 25,
  city: "New York"
});

// Insert multiple users at once
db.users.insertMany([
  { name: "Bob", age: 30, city: "Chicago" },
  { name: "Charlie", age: 35, city: "Boston" }
]);
```

### Ordered vs unordered inserts

By default `insertMany()` is **ordered**: if document #3 fails, documents #4 onward are never attempted. Pass `{ ordered: false }` to keep going past failures:

```javascript
db.users.insertMany(
  [ /* ...docs... */ ],
  { ordered: false }
);
```

---

## 2. Read Operations (the missing "R")

Reads weren't in the original notes, but CRUD is incomplete without them.

| Method | Returns |
|---|---|
| `find(filter)` | A **cursor** over all matching documents |
| `findOne(filter)` | A single document (or `null`) |
| `countDocuments(filter)` | The number of matches |

```javascript
// Everyone
db.users.find();

// Everyone over 30, sorted youngest first, limited to 5
db.users.find({ age: { $gt: 30 } }).sort({ age: 1 }).limit(5);

// Projection: return only name and city, suppress _id
db.users.find({ city: "Boston" }, { name: 1, city: 1, _id: 0 });

// Readable output in the shell
db.users.find().pretty();
```

### Common query operators

| Operator | Meaning |
|---|---|
| `$eq` / `$ne` | Equals / not equals |
| `$gt` `$gte` `$lt` `$lte` | Greater / less than comparisons |
| `$in` / `$nin` | Value is (or isn't) in an array of options |
| `$exists` | Field is present or absent |
| `$regex` | Pattern match on a string |
| `$and` `$or` `$not` | Logical combinators |

```javascript
db.users.find({ city: { $in: ["Boston", "Chicago"] } });
db.users.find({ $or: [ { age: { $lt: 25 } }, { city: "New York" } ] });
```

> **SQL parallel:** the filter object is your `WHERE` clause, the projection object is your `SELECT` list, and `.sort()` / `.limit()` are `ORDER BY` / `LIMIT`.

---

## 3. Update Operations

Update methods take two arguments: a **query filter** to target documents, and an **update document** containing operators that change specific fields without wiping out the rest.

| Method | Behaviour |
|---|---|
| `updateOne(filter, update)` | Modifies the **first** matching document |
| `updateMany(filter, update)` | Modifies **all** matching documents |
| `replaceOne(filter, doc)` | Discards the existing document entirely and swaps in a new one (keeping the original `_id`) |
| `findOneAndUpdate(filter, update)` | Updates and **returns** the document — useful in app code |

### Common update operators

| Operator | Effect |
|---|---|
| `$set` | Sets a field's value, creating it if missing |
| `$inc` | Increments/decrements a numeric value |
| `$unset` | Removes a field from the document |
| `$rename` | Renames a field |
| `$mul` | Multiplies a numeric value |
| `$min` / `$max` | Only updates if the new value is lower / higher |
| `$currentDate` | Sets a field to the current date |

```javascript
// Update Alice's city (first match only)
db.users.updateOne(
  { name: "Alice" },
  { $set: { city: "San Francisco" } }
);

// Give everyone in Chicago a birthday
db.users.updateMany(
  { city: "Chicago" },
  { $inc: { age: 1 } }
);

// Drop a field entirely
db.users.updateOne(
  { name: "Bob" },
  { $unset: { city: "" } }   // the value is ignored
);
```

> ⚠️ **Forgetting the operator is the classic beginner bug.** `db.users.updateOne({ name: "Alice" }, { city: "SF" })` throws an error in modern drivers — but in older shells it silently *replaced* the whole document, deleting `name` and `age`. Always wrap the change in `$set`.

### Array update operators

| Operator | Effect |
|---|---|
| `$push` | Appends a value to an array |
| `$addToSet` | Appends only if not already present |
| `$pull` | Removes all values matching a condition |
| `$pop` | Removes first (`-1`) or last (`1`) element |

```javascript
db.users.updateOne(
  { name: "Alice" },
  { $push: { hobbies: "cycling" } }
);

db.users.updateOne(
  { name: "Alice" },
  { $addToSet: { hobbies: "cycling" } }   // no duplicate
);
```

### What is an "upsert"?

Pass `{ upsert: true }` as a third argument. If a document matches the filter, it updates normally. If nothing matches, MongoDB **inserts** a new document built from the filter criteria plus the update parameters.

```javascript
db.users.updateOne(
  { name: "Diana" },
  { $set: { city: "Seattle", age: 28 } },
  { upsert: true }
);
// No Diana existed → creates { name: "Diana", city: "Seattle", age: 28 }
```

This is MongoDB's equivalent of `MERGE` / `INSERT ... ON DUPLICATE KEY UPDATE` — very handy for sync jobs and idempotent writes.

---

## 4. Delete Operations

These permanently remove documents matching a query filter.

| Method | Behaviour |
|---|---|
| `deleteOne(filter)` | Removes the **first** matching document |
| `deleteMany(filter)` | Removes **all** matching documents |
| `findOneAndDelete(filter)` | Removes and returns the document |

```javascript
// Delete a specific user
db.users.deleteOne({ name: "Alice" });

// Delete everyone under 30
db.users.deleteMany({ age: { $lt: 30 } });
```

> ⚠️ **`deleteMany({})` with an empty filter wipes every document in the collection.** There is no confirmation prompt and no transaction to roll back. To remove the collection *and* its indexes instead, use `db.users.drop()`.

---

## Understanding the result object

Every write method returns a result you should actually check in application code:

```javascript
const res = await db.users.updateOne({ name: "Alice" }, { $set: { age: 26 } });
// res.matchedCount   → how many documents the filter found
// res.modifiedCount  → how many were actually changed
// res.upsertedId     → the _id if an upsert created a document
```

`matchedCount: 1, modifiedCount: 0` means you found the document but set a field to the value it already had — a common source of "why didn't my update work?" confusion.

---

## Summary cheatsheet

| Operation | Single-document | Multi-document |
|---|---|---|
| **Create** | `db.coll.insertOne(doc)` | `db.coll.insertMany([doc1, doc2])` |
| **Read** | `db.coll.findOne(filter)` | `db.coll.find(filter)` |
| **Update** | `db.coll.updateOne(filter, update)` | `db.coll.updateMany(filter, update)` |
| **Replace** | `db.coll.replaceOne(filter, doc)` | — |
| **Delete** | `db.coll.deleteOne(filter)` | `db.coll.deleteMany(filter)` |

---

## Shell tips

```javascript
show dbs              // list all databases
use myapp             // switch database (creates lazily on first write)
show collections      // list collections in current db
db.users.countDocuments()
db.users.drop()       // delete the whole collection
```

Set a custom prompt in `~/.mongoshrc.js` so you always know where you are:

```javascript
prompt = function() {
  return db.getName() + " @ " + db.getMongo().host + "> ";
};
```